# Binarization

**Binarization** is a data preprocessing technique that converts numerical values into **only two possible values**, usually:

* **0** (False, No, Off)
* **1** (True, Yes, On)

It uses a **threshold** (cutoff value) to decide whether a value becomes 0 or 1.

### Rule

If the value is **greater than the threshold**, assign **1**.

If the value is **less than or equal to the threshold**, assign **0**.

Mathematically:

x={1,  ​if x>threshold
  {0,  if x≤threshold​

---

# Beginner Example

Suppose we have exam marks:

| Student | Marks |
| ------- | ----: |
| A       |    20 |
| B       |    35 |
| C       |    50 |
| D       |    65 |
| E       |    80 |

Choose a **threshold = 50**.

### Apply the rule

* Marks > 50 → 1
* Marks ≤ 50 → 0

| Marks | Binary Output |
| ----: | ------------: |
|    20 |             0 |
|    35 |             0 |
|    50 |             0 |
|    65 |             1 |
|    80 |             1 |

The numerical values have now been converted into **binary values (0 and 1)**.

---

# Real-Life Example

Imagine an online shopping website that wants to identify **high spenders**.

Customer spending:

| Customer | Amount Spent ($) |
| -------- | ---------------: |
| A        |               50 |
| B        |              120 |
| C        |              300 |
| D        |               80 |
| E        |              450 |

Threshold = **100**

Rule:

* Spending > 100 → High Spender (1)
* Spending ≤ 100 → Not High Spender (0)

| Amount | Binary |
| -----: | -----: |
|     50 |      0 |
|    120 |      1 |
|    300 |      1 |
|     80 |      0 |
|    450 |      1 |

---

# Why Use Binarization?

* Converts numerical data into **Yes/No** or **True/False** values.
* Makes data simpler for some machine learning algorithms.
* Useful for creating binary features (e.g., *Passed/Failed*, *Spam/Not Spam*, *Purchased/Not Purchased*).

---

# Difference Between Binarization and Binning

| Feature          | Binarization   | Binning                             |
| ---------------- | -------------- | ----------------------------------- |
| Number of groups | **2** (0 or 1) | **Multiple** bins                   |
| Uses threshold?  | Yes            | Not necessarily (depends on method) |
| Output           | Binary values  | Categories or intervals             |
| Example          | 0, 1           | Low, Medium, High                   |


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer

In [2]:
df = sns.load_dataset('titanic')
df = df[['age', 'fare', 'sibsp', 'parch', 'survived']]
df.head()

,age,fare,sibsp,parch,survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [3]:
df.dropna(inplace=True)

In [4]:
df['family'] = df['sibsp'] + df['parch']

In [5]:
df.head()

,age,fare,sibsp,parch,survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [6]:
df.drop(columns=['sibsp', 'parch'], inplace=True)

In [7]:
df.head()

,age,fare,survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [9]:
X = df.drop(columns=['survived'])
y = df['survived']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [11]:
X_train.head()

,age,fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [12]:
# Without Binarization 
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

accuracy_score(y_test, y_pred)

0.6293706293706294

In [13]:
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy'))

np.float64(0.6485524256651017)

In [14]:
# Applying Binarization 
from sklearn.preprocessing import Binarizer

In [16]:
trf = ColumnTransformer([
    ('bin', Binarizer(copy=False), ['family'])
], remainder='passthrough')

In [19]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [20]:
pd.DataFrame(X_train_trf, columns=['family', 'age', 'fare'])

,family,age,fare
0,1.0,31.0,20.5250
1,1.0,26.0,14.4542
2,1.0,30.0,16.1000
3,0.0,33.0,7.7750
4,0.0,25.0,13.0000
...,...,...,...
566,1.0,46.0,61.1750
567,0.0,25.0,13.0000
568,0.0,41.0,134.5000
569,1.0,33.0,20.5250


In [21]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf, y_train)
y_pred2 = clf.predict(X_test_trf)
accuracy_score(y_test, y_pred2)

0.6293706293706294

In [22]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy'))

np.float64(0.6276017214397496)